# Bike Sharing Demand Prediction
**Project 3 – Predictive Analytics Using Historical Data**

**Business question:** Can we predict daily bike rental demand from historical calendar, seasonal, and weather information?

**Dataset:** UCI Bike Sharing Dataset (`day.csv`). The target is `cnt` (total rental count).

**Method:** Chronological holdout evaluation. The last 20% of dates are held out for testing; earlier dates are used for training. We compare a Linear Regression baseline with a Random Forest model.

> Important: `casual` and `registered` are excluded from the predictors because `cnt = casual + registered`. Using them would leak the target into the model. The dataset contains only 2011–2012, so this is a historical holdout forecasting exercise, not a guarantee of current real-world demand.


## 1. Upload the dataset

In [ ]:
from google.colab import files
uploaded = files.upload()  # Select day.csv


## 2. Import libraries and load data

In [ ]:
import io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style="whitegrid")


In [ ]:
# Load the uploaded daily dataset
file_name = next((name for name in uploaded if name.lower().endswith(".csv")), None)
if file_name is None:
    raise ValueError("Please upload day.csv")

df = pd.read_csv(io.BytesIO(uploaded[file_name]))
print("Dataset shape:", df.shape)
display(df.head())
df.info()


## 3. Data quality inspection

In [ ]:
print("Missing values by column:")
display(df.isna().sum().to_frame("missing_values"))

print("Duplicate rows:", df.duplicated().sum())
print("Date range:", df["dteday"].min(), "to", df["dteday"].max())
print("Target summary:")
display(df["cnt"].describe())


## 4. Data cleaning and feature engineering

In [ ]:
# Work on a copy and parse the date
data = df.copy()
data["dteday"] = pd.to_datetime(data["dteday"], errors="coerce")

# Remove rows with invalid dates or missing target values, if any
data = data.dropna(subset=["dteday", "cnt"]).copy()
data = data.drop_duplicates().sort_values("dteday").reset_index(drop=True)

# Calendar features derived from the date
data["date_year"] = data["dteday"].dt.year
data["date_month"] = data["dteday"].dt.month
data["date_day"] = data["dteday"].dt.day
data["date_dayofweek"] = data["dteday"].dt.dayofweek
data["date_dayofyear"] = data["dteday"].dt.dayofyear

# Target is total rental count
target = "cnt"

# Exclude identifiers, raw date, and casual/registered to prevent leakage:
# cnt is the sum of casual + registered.
feature_cols = [
    "season", "yr", "mnth", "holiday", "weekday", "workingday",
    "weathersit", "temp", "atemp", "hum", "windspeed",
    "date_month", "date_day", "date_dayofweek", "date_dayofyear"
]

X = data[feature_cols].copy()
y = data[target].astype(float)

print("Cleaned data shape:", data.shape)
print("Predictor count:", len(feature_cols))
print("Target:", target)


## 5. Exploratory data analysis

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(data["dteday"], data["cnt"], linewidth=1)
plt.title("Daily Bike Rental Demand Over Time")
plt.xlabel("Date")
plt.ylabel("Total Rentals (cnt)")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(data=data, x="season", y="cnt")
plt.title("Daily Bike Rentals by Season")
plt.xlabel("Season (1 = Spring, 2 = Summer, 3 = Fall, 4 = Winter)")
plt.ylabel("Total Rentals")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=data, x="temp", y="cnt", alpha=0.65)
plt.title("Temperature vs Daily Bike Rentals")
plt.xlabel("Normalized Temperature")
plt.ylabel("Total Rentals")
plt.tight_layout()
plt.show()


In [ ]:
monthly = data.groupby("mnth", as_index=False)["cnt"].mean()
plt.figure(figsize=(9, 5))
sns.lineplot(data=monthly, x="mnth", y="cnt", marker="o")
plt.title("Average Daily Bike Rentals by Month")
plt.xlabel("Month")
plt.ylabel("Average Rentals")
plt.xticks(range(1, 13))
plt.tight_layout()
plt.show()


## 6. Chronological train-test split

In [ ]:
# Use the earliest 80% of dates for training and the latest 20% for testing.
split_index = int(len(data) * 0.80)

X_train, X_test = X.iloc[:split_index], X.iloc[split_index:]
y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]
test_dates = data["dteday"].iloc[split_index:]

print("Training period:", data["dteday"].iloc[0].date(), "to", data["dteday"].iloc[split_index - 1].date())
print("Testing period:", data["dteday"].iloc[split_index].date(), "to", data["dteday"].iloc[-1].date())
print("Training records:", len(X_train))
print("Testing records:", len(X_test))


## 7. Build and compare prediction models

In [ ]:
categorical_features = [
    "season", "yr", "mnth", "holiday", "weekday", "workingday", "weathersit",
    "date_month", "date_day", "date_dayofweek"
]
numeric_features = ["temp", "atemp", "hum", "windspeed", "date_dayofyear"]

# Linear Regression pipeline: impute, scale numeric variables, one-hot encode categories.
linear_preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), numeric_features),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), categorical_features)
    ]
)

linear_model = Pipeline([
    ("preprocessor", linear_preprocessor),
    ("model", LinearRegression())
])

# Random Forest pipeline: impute numeric and categorical columns, then fit trees.
rf_preprocessor = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), numeric_features),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), categorical_features)
    ]
)

random_forest = Pipeline([
    ("preprocessor", rf_preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=400,
        max_depth=8,
        min_samples_leaf=3,
        random_state=42,
        n_jobs=-1
    ))
])

models = {
    "Linear Regression": linear_model,
    "Random Forest": random_forest
}

predictions = {}
results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    pred = np.maximum(pred, 0)  # rental counts cannot be negative
    predictions[name] = pred

    mae = mean_absolute_error(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    r2 = r2_score(y_test, pred)

    results.append({
        "Model": name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    })

results_df = pd.DataFrame(results).sort_values("RMSE").reset_index(drop=True)
display(results_df)


## 8. Select the best model and evaluate

In [ ]:
best_model_name = results_df.loc[0, "Model"]
best_predictions = predictions[best_model_name]

print("Best model by lowest test RMSE:", best_model_name)
print(f"MAE:  {mean_absolute_error(y_test, best_predictions):,.2f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, best_predictions)):,.2f}")
print(f"R²:   {r2_score(y_test, best_predictions):.3f}")


## 9. Actual vs predicted visualization

In [ ]:
comparison = pd.DataFrame({
    "Date": test_dates.values,
    "Actual": y_test.values,
    "Predicted": best_predictions
})

plt.figure(figsize=(13, 5))
plt.plot(comparison["Date"], comparison["Actual"], label="Actual", linewidth=2)
plt.plot(comparison["Date"], comparison["Predicted"], label="Predicted", linewidth=2)
plt.title(f"Actual vs Predicted Daily Bike Rentals – {best_model_name}")
plt.xlabel("Date")
plt.ylabel("Total Rentals")
plt.legend()
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

display(comparison.head(10))


## 10. Prediction error analysis

In [ ]:
comparison["Error"] = comparison["Actual"] - comparison["Predicted"]

plt.figure(figsize=(8, 5))
sns.scatterplot(data=comparison, x="Actual", y="Predicted", alpha=0.7)
lims = [
    min(comparison["Actual"].min(), comparison["Predicted"].min()),
    max(comparison["Actual"].max(), comparison["Predicted"].max())
]
plt.plot(lims, lims, "--", label="Perfect prediction")
plt.title("Actual vs Predicted Values")
plt.xlabel("Actual Rentals")
plt.ylabel("Predicted Rentals")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 5))
sns.histplot(comparison["Error"], bins=25, kde=True)
plt.title("Prediction Error Distribution")
plt.xlabel("Actual - Predicted")
plt.tight_layout()
plt.show()


## 11. Export results

In [ ]:
# Save test predictions and model comparison for your project documentation.
comparison.to_csv("bike_rental_test_predictions.csv", index=False)
results_df.to_csv("model_evaluation_results.csv", index=False)

print("Saved:")
print("- bike_rental_test_predictions.csv")
print("- model_evaluation_results.csv")


## 12. Conclusion – complete after running

After running all cells, use the measured values in `results_df` to write the final conclusion. Report the best model based on the test RMSE, its MAE, RMSE and R², and explain where predictions differ from actual rentals.

**Do not claim future operational forecasting beyond the test period without additional future weather and calendar inputs.** The dataset documentation describes observations from 2011–2012.
